In [5]:
##시군구별 성장 쇠퇴 상권
# 1~2월 시군구별 총 이용금액 + 건수
early_ccg = (
    df[df['STRD_YYMM'].isin([202601, 202602])]
    .groupby('CCG_NM')
    .agg(amt_early=('amt', 'sum'), cnt_early=('cnt', 'sum'))
    .reset_index()
)

# 5~6월 시군구별 총 이용금액 + 건수
late_ccg = (
    df[df['STRD_YYMM'].isin([202605, 202606])]
    .groupby('CCG_NM')
    .agg(amt_late=('amt', 'sum'), cnt_late=('cnt', 'sum'))
    .reset_index()
)

# 비교
ccg_compare = pd.merge(early_ccg, late_ccg, on='CCG_NM')

# 증감률
ccg_compare['change_rate(%)'] = (
    (ccg_compare['amt_late'] - ccg_compare['amt_early'])
    / ccg_compare['amt_early'] * 100
).round(2)

# 규모 파악용: 해당 시군구의 전체 기간 총 거래건수
ccg_compare['total_cnt'] = ccg_compare['cnt_early'] + ccg_compare['cnt_late']

display(ccg_compare.sort_values('change_rate(%)', ascending=False))

,CCG_NM,amt_early,cnt_early,amt_late,cnt_late,change_rate(%),total_cnt
156,울릉군,580940000,23166,1590930000,58853,173.85,82019
141,영양군,644070000,30657,1303990000,57791,102.46,88448
21,곡성군,1331640000,80253,2562460000,141222,92.43,221475
172,장수군,1111310000,53655,2085440000,98604,87.66,152259
221,합천군,2093580000,82694,3377670000,136701,61.33,219395
...,...,...,...,...,...,...,...
13,계양구,24936400000,1341937,23708620000,1388002,-4.92,2729939
125,양산시,37655350000,1865179,35772520000,1908187,-5.00,3773366
136,영덕군,5258600000,176831,4681950000,185075,-10.97,361906
211,평창군,5926240000,247293,5233780000,220132,-11.68,467425


In [6]:
# 거래 건수 분포 확인 (기준선 정하기 위해)
display(ccg_compare['total_cnt'].describe())

count    2.330000e+02
mean     2.690890e+06
std      3.184691e+06
min      8.201900e+04
25%      4.196650e+05
50%      2.063762e+06
75%      3.877907e+06
max      2.104727e+07
Name: total_cnt, dtype: float64

In [7]:
# 표본이 너무 작은 시군구는 제외 (기준값은 위 describe 결과 보고 같이 정하기)
threshold = ccg_compare['total_cnt'].quantile(0.25)  # 하위 25% 제외 (예시)

reliable_ccg = ccg_compare[ccg_compare['total_cnt'] >= threshold]

print("성장 상권 TOP 10")
display(reliable_ccg.sort_values('change_rate(%)', ascending=False).head(10))

print("쇠퇴 상권 TOP 10")
display(reliable_ccg.sort_values('change_rate(%)').head(10))

성장 상권 TOP 10


,CCG_NM,amt_early,cnt_early,amt_late,cnt_late,change_rate(%),total_cnt
208,태안군,5961150000,239307,9553450000,386838,60.26,626145
51,남해군,4418090000,168642,6940910000,265504,57.10,434146
15,고성군,6219920000,277000,9367710000,415351,50.61,692351
0,가평군,6775420000,278784,9868570000,395984,45.65,674768
9,거창군,3572360000,172012,4984860000,247653,39.54,419665
137,영도구,12248410000,699481,16974780000,1004523,38.59,1704004
193,창원시 진해구,20396520000,1048631,26880710000,1499111,31.79,2547742
190,창원시 마산회원구,20106870000,1054324,26319040000,1550768,30.90,2605092
88,사하구,33886470000,2029708,43992960000,2738301,29.82,4768009
192,창원시 의창구,25404120000,1246449,32849660000,1770985,29.31,3017434


쇠퇴 상권 TOP 10


,CCG_NM,amt_early,cnt_early,amt_late,cnt_late,change_rate(%),total_cnt
211,평창군,5926240000,247293,5233780000,220132,-11.68,467425
125,양산시,37655350000,1865179,35772520000,1908187,-5.00,3773366
13,계양구,24936400000,1341937,23708620000,1388002,-4.92,2729939
48,남동구,44103200000,2363249,42685970000,2478070,-3.21,4841319
224,홍성군,8307370000,331477,8050740000,390122,-3.09,721599
84,부평구,44175260000,2459598,43200550000,2599692,-2.21,5059290
132,연수구,36627550000,1895844,36146420000,2066115,-1.31,3961959
81,부천시 소사구,19589130000,1044827,19338270000,1145838,-1.28,2190665
40,금천구,28850280000,1896318,28515130000,2045988,-1.16,3942306
1,강남구,112841710000,6724460,111911180000,6978515,-0.82,13702975


In [8]:
# 규모 구간 태깅 (25%, 50%, 75% 분위 기준)
def size_tier(cnt):
    if cnt < 419665:
        return '소규모'
    elif cnt < 2063762:
        return '중소규모'
    elif cnt < 3877907:
        return '중대규모'
    else:
        return '대규모'

ccg_compare['size_tier'] = ccg_compare['total_cnt'].apply(size_tier)

print("성장 상권 TOP 15 (규모 태그 포함)")
display(
    ccg_compare.sort_values('change_rate(%)', ascending=False)
    [['CCG_NM', 'change_rate(%)', 'total_cnt', 'size_tier']]
    .head(15)
)

print("\n쇠퇴 상권 TOP 15 (규모 태그 포함)")
display(
    ccg_compare.sort_values('change_rate(%)')
    [['CCG_NM', 'change_rate(%)', 'total_cnt', 'size_tier']]
    .head(15)
)

성장 상권 TOP 15 (규모 태그 포함)


,CCG_NM,change_rate(%),total_cnt,size_tier
156,울릉군,173.85,82019,소규모
141,영양군,102.46,88448,소규모
21,곡성군,92.43,221475,소규모
172,장수군,87.66,152259,소규모
221,합천군,61.33,219395,소규모
142,영월군,60.88,215170,소규모
208,태안군,60.26,626145,중소규모
149,옹진군,59.01,193647,소규모
199,청양군,58.04,176333,소규모
51,남해군,57.10,434146,중소규모



쇠퇴 상권 TOP 15 (규모 태그 포함)


,CCG_NM,change_rate(%),total_cnt,size_tier
70,무주군,-30.58,189450,소규모
211,평창군,-11.68,467425,중소규모
136,영덕군,-10.97,361906,소규모
125,양산시,-5.00,3773366,중대규모
13,계양구,-4.92,2729939,중대규모
48,남동구,-3.21,4841319,대규모
224,홍성군,-3.09,721599,중소규모
84,부평구,-2.21,5059290,대규모
132,연수구,-1.31,3961959,대규모
81,부천시 소사구,-1.28,2190665,중대규모


In [9]:
# 1) 인천 지역 쏠림 확인 - SIDO_NM 붙여서 보기
ccg_sido = df[['SIDO_NM', 'CCG_NM']].drop_duplicates()
decline_with_sido = ccg_compare.sort_values('change_rate(%)').head(15).merge(ccg_sido, on='CCG_NM')
display(decline_with_sido[['SIDO_NM', 'CCG_NM', 'change_rate(%)', 'total_cnt']])

,SIDO_NM,CCG_NM,change_rate(%),total_cnt
0,전북특별자치도,무주군,-30.58,189450
1,강원특별자치도,평창군,-11.68,467425
2,경상북도,영덕군,-10.97,361906
3,경상남도,양산시,-5.00,3773366
4,인천광역시,계양구,-4.92,2729939
5,인천광역시,남동구,-3.21,4841319
6,충청남도,홍성군,-3.09,721599
7,인천광역시,부평구,-2.21,5059290
8,인천광역시,연수구,-1.31,3961959
9,경기도,부천시 소사구,-1.28,2190665


In [10]:
# 2) 성장 TOP 상위 5개 소규모 지역의 업종 구성 - 정말 관광/외식 업종이 주도했는지 확인
top_growth_regions = ['울릉군', '영양군', '곡성군', '장수군', '합천군']

for region in top_growth_regions:
    region_data = df[df['CCG_NM'] == region]
    early = region_data[region_data['STRD_YYMM'].isin([202601,202602])].groupby('TP_BUZ_NM')['amt'].sum()
    late = region_data[region_data['STRD_YYMM'].isin([202605,202606])].groupby('TP_BUZ_NM')['amt'].sum()
    change = ((late - early) / early * 100).round(1).sort_values(ascending=False)
    print(f"\n[{region}] 업종별 증감률")
    print(change)


[울릉군] 업종별 증감률
TP_BUZ_NM
서양음식     335.4
일반한식     233.2
스넥       201.0
중국음식     142.7
편 의 점     79.5
슈퍼 마켓      8.7
일식회집       NaN
제 과 점      NaN
Name: amt, dtype: float64

[영양군] 업종별 증감률
TP_BUZ_NM
일식회집     276.7
슈퍼 마켓    129.0
스넥       110.4
제 과 점    102.5
서양음식      96.7
중국음식      93.3
편 의 점     88.8
일반한식      86.1
Name: amt, dtype: float64

[곡성군] 업종별 증감률
TP_BUZ_NM
일식회집     152.1
서양음식     139.4
제 과 점    130.6
슈퍼 마켓    121.8
일반한식     104.9
중국음식      77.7
편 의 점     48.6
스넥        48.1
Name: amt, dtype: float64

[장수군] 업종별 증감률
TP_BUZ_NM
제 과 점    187.4
중국음식     153.0
스넥       143.6
슈퍼 마켓     98.9
서양음식      84.0
일반한식      79.0
편 의 점     65.3
일식회집      61.5
Name: amt, dtype: float64

[합천군] 업종별 증감률
TP_BUZ_NM
갈비전문점    87.9
스넥       77.8
일반한식     70.7
슈퍼 마켓    63.2
제 과 점    62.7
서양음식     58.8
편 의 점    57.8
중국음식     46.4
대형할인점    -7.4
일식회집    -40.6
Name: amt, dtype: float64


In [11]:
# 시도별 평균 증감률 (모든 시도 동일 기준으로 비교)
ccg_sido_all = ccg_compare.merge(ccg_sido, on='CCG_NM')

sido_avg_change = (
    ccg_sido_all.groupby('SIDO_NM')['change_rate(%)']
    .mean()
    .round(2)
    .sort_values()
)

display(sido_avg_change)

SIDO_NM
서울특별시       5.86
경기도         9.53
인천광역시      10.03
대전광역시      11.11
세종특별자치시    12.04
광주광역시      13.22
울산광역시      14.30
제주특별자치도    15.30
충청북도       15.72
대구광역시      15.79
강원특별자치도    17.46
부산광역시      17.75
전라남도       18.78
충청남도       19.34
경상북도       22.43
전북특별자치도    22.94
경상남도       27.00
Name: change_rate(%), dtype: float64

In [12]:
# 수도권 vs 지방 구분
capital_area = ['서울특별시', '경기도', '인천광역시']

df['region_type'] = df['SIDO_NM'].apply(
    lambda x: '수도권' if x in capital_area else '지방'
)

# 지역군 x 연령대 x 업종별 증감률 계산
def calc_growth(data, group_cols):
    early = (
        data[data['STRD_YYMM'].isin([202601, 202602])]
        .groupby(group_cols)['amt'].sum()
    )
    late = (
        data[data['STRD_YYMM'].isin([202605, 202606])]
        .groupby(group_cols)['amt'].sum()
    )
    result = pd.concat([early, late], axis=1, keys=['early', 'late'])
    result['change_rate(%)'] = ((result['late'] - result['early']) / result['early'] * 100).round(2)
    return result.reset_index()

cross_result = calc_growth(df, ['region_type', 'AGE_CD', 'TP_BUZ_NM'])

# 가설 핵심 대상만 필터: 슈퍼마켓 + 대형할인점, 전 연령대 비교
target_industries = ['슈퍼 마켓', '대형할인점']
focus = cross_result[cross_result['TP_BUZ_NM'].isin(target_industries)]

# 지역군 x 연령대 별로 피벗해서 한눈에 비교
pivot_focus = focus.pivot_table(
    index=['TP_BUZ_NM', 'AGE_CD'],
    columns='region_type',
    values='change_rate(%)'
)

display(pivot_focus)

region_type         수도권      지방
TP_BUZ_NM AGE_CD               
대형할인점     1      -24.74  -23.01
          2      -15.90  -14.25
          3      -12.24  -12.09
          4      -13.84  -13.98
          5      -12.09  -12.34
          6       -6.01   -3.38
          x        2.13    2.47
슈퍼 마켓     1       -3.90    7.91
          2       -3.63    2.40
          3       -3.51   -1.67
          4       -5.81   -5.15
          5       -5.53   -3.03
          6       -1.29    4.41
          x       -1.62  217.71

In [13]:
# 슈퍼마켓 한정, 지역x연령별 실제 소비 규모(early/late 절대값) 확인
supermarket = df[df['TP_BUZ_NM'] == '슈퍼 마켓']

early_sm = (
    supermarket[supermarket['STRD_YYMM'].isin([202601, 202602])]
    .groupby(['region_type', 'AGE_CD'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)

late_sm = (
    supermarket[supermarket['STRD_YYMM'].isin([202605, 202606])]
    .groupby(['region_type', 'AGE_CD'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)

scale_check = pd.merge(early_sm, late_sm, on=['region_type', 'AGE_CD'])
scale_check['change_rate(%)'] = (
    (scale_check['amt_late'] - scale_check['amt_early']) / scale_check['amt_early'] * 100
).round(2)
scale_check['amt_diff'] = scale_check['amt_late'] - scale_check['amt_early']

# 보기 편하게 정렬 (지역x연령 순)
display(scale_check.sort_values(['region_type', 'AGE_CD']))

,region_type,AGE_CD,amt_early,amt_late,change_rate(%),amt_diff
0,수도권,1,1135870000,1091600000,-3.90,-44270000
1,수도권,2,15862390000,15286940000,-3.63,-575450000
2,수도권,3,39568290000,38179190000,-3.51,-1389100000
3,수도권,4,68259980000,64293130000,-5.81,-3966850000
4,수도권,5,101854460000,96225520000,-5.53,-5628940000
5,수도권,6,149487090000,147560970000,-1.29,-1926120000
6,수도권,x,6314530000,6212270000,-1.62,-102260000
7,지방,1,2050740000,2213020000,7.91,162280000
8,지방,2,26630180000,27268810000,2.40,638630000
9,지방,3,57807160000,56841070000,-1.67,-966090000


In [2]:
import pandas as pd

df = pd.read_csv(r'C:\Users\yeseo\Downloads\ODR20260914210047380\ABP_CONTEST_DATA.csv')

In [14]:
# 60대 이상(AGE_CD=6) 데이터만 필터
sixties = df[df['AGE_CD'] == '6']

# 업종별 1월 vs 6월 비교
early_60 = (
    sixties[sixties['STRD_YYMM'] == 202601]
    .groupby('TP_BUZ_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_jan'})
)

late_60 = (
    sixties[sixties['STRD_YYMM'] == 202606]
    .groupby('TP_BUZ_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_jun'})
)

compare_60 = pd.merge(early_60, late_60, on='TP_BUZ_NM')
compare_60['change_rate(%)'] = (
    (compare_60['amt_jun'] - compare_60['amt_jan']) / compare_60['amt_jan'] * 100
).round(2)
compare_60['amt_diff'] = compare_60['amt_jun'] - compare_60['amt_jan']

display(compare_60.sort_values('change_rate(%)', ascending=False))

,TP_BUZ_NM,amt_jan,amt_jun,change_rate(%),amt_diff
10,한정식,93130000,115380000,23.89,22250000
4,스넥,16967800000,20218390000,19.16,3250590000
9,편 의 점,53027030000,63186280000,19.16,10159250000
2,서양음식,43934270000,49311580000,12.24,5377310000
5,일반한식,282895440000,305275110000,7.91,22379670000
6,일식회집,14661290000,15396030000,5.01,734740000
3,슈퍼 마켓,155803120000,159856750000,2.60,4053630000
0,갈비전문점,645330000,658650000,2.06,13320000
7,제 과 점,18229290000,18171840000,-0.32,-57450000
8,중국음식,17459150000,16943000000,-2.96,-516150000


In [15]:
early_60_avg = (
    sixties[sixties['STRD_YYMM'].isin([202601, 202602])]
    .groupby('TP_BUZ_NM')['amt'].sum()
    .reset_index().rename(columns={'amt': 'amt_early'})
)
late_60_avg = (
    sixties[sixties['STRD_YYMM'].isin([202605, 202606])]
    .groupby('TP_BUZ_NM')['amt'].sum()
    .reset_index().rename(columns={'amt': 'amt_late'})
)
compare_60_avg = pd.merge(early_60_avg, late_60_avg, on='TP_BUZ_NM')
compare_60_avg['change_rate(%)'] = (
    (compare_60_avg['amt_late'] - compare_60_avg['amt_early']) / compare_60_avg['amt_early'] * 100
).round(2)
compare_60_avg['amt_diff'] = compare_60_avg['amt_late'] - compare_60_avg['amt_early']

display(compare_60_avg.sort_values('amt_diff', ascending=False))

,TP_BUZ_NM,amt_early,amt_late,change_rate(%),amt_diff
5,일반한식,531785970000,637183050000,19.82,105397080000
9,편 의 점,102877390000,126258000000,22.73,23380610000
2,서양음식,84315500000,101864530000,20.81,17549030000
4,스넥,32147090000,41231420000,28.26,9084330000
3,슈퍼 마켓,318637350000,324174850000,1.74,5537500000
6,일식회집,27708900000,32776200000,18.29,5067300000
8,중국음식,33740510000,36031320000,6.79,2290810000
7,제 과 점,37484290000,38629740000,3.06,1145450000
0,갈비전문점,1180230000,1405570000,19.09,225340000
10,한정식,151000000,214120000,41.80,63120000


In [16]:
# 60대 + 일반한식만 필터
sixties_korean = sixties[sixties['TP_BUZ_NM'] == '일반한식']

# 1~2월
early_korean = (
    sixties_korean[sixties_korean['STRD_YYMM'].isin([202601, 202602])]
    .groupby('SIDO_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)

# 5~6월
late_korean = (
    sixties_korean[sixties_korean['STRD_YYMM'].isin([202605, 202606])]
    .groupby('SIDO_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)

# 비교
compare_korean_region = pd.merge(
    early_korean,
    late_korean,
    on='SIDO_NM'
)

compare_korean_region['change_rate(%)'] = (
    (compare_korean_region['amt_late'] - compare_korean_region['amt_early'])
    / compare_korean_region['amt_early'] * 100
).round(2)

compare_korean_region['amt_diff'] = (
    compare_korean_region['amt_late']
    - compare_korean_region['amt_early']
)

display(
    compare_korean_region.sort_values(
        'amt_diff',
        ascending=False
    )
)

,SIDO_NM,amt_early,amt_late,change_rate(%),amt_diff
1,경기도,115559500000,141184820000,22.18,25625320000
8,서울특별시,90845890000,103196380000,13.59,12350490000
7,부산광역시,62643980000,72372380000,15.53,9728400000
5,대구광역시,31518580000,38373140000,21.75,6854560000
3,경상북도,29324660000,35969470000,22.66,6644810000
0,강원특별자치도,17974610000,24351190000,35.48,6376580000
2,경상남도,36727500000,42705300000,16.28,5977800000
15,충청남도,18435630000,24373270000,32.21,5937640000
12,전라남도,21483110000,27181780000,26.53,5698670000
16,충청북도,15549050000,20520050000,31.97,4971000000


In [17]:
# 60대 + 일반한식 월별 소비금액
monthly_60_korean = (
    sixties[sixties['TP_BUZ_NM'] == '일반한식']
    .groupby('STRD_YYMM')['amt']
    .sum()
    .reset_index()
)

display(monthly_60_korean)

,STRD_YYMM,amt
0,202601,282895440000
1,202602,248890530000
2,202603,295944060000
3,202604,301661240000
4,202605,331907940000
5,202606,305275110000


In [18]:
# 일반한식 연령대별 1~2월 → 5~6월 소비금액 변화

korean_food = df[df['TP_BUZ_NM'] == '일반한식']

early_age = (
    korean_food[korean_food['STRD_YYMM'].isin([202601, 202602])]
    .groupby('AGE_CD')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)

late_age = (
    korean_food[korean_food['STRD_YYMM'].isin([202605, 202606])]
    .groupby('AGE_CD')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)

compare_korean_age = pd.merge(
    early_age,
    late_age,
    on='AGE_CD'
)

compare_korean_age['change_rate(%)'] = (
    (compare_korean_age['amt_late'] - compare_korean_age['amt_early'])
    / compare_korean_age['amt_early'] * 100
).round(2)

compare_korean_age['amt_diff'] = (
    compare_korean_age['amt_late']
    - compare_korean_age['amt_early']
)

display(
    compare_korean_age.sort_values('change_rate(%)', ascending=False)
)

,AGE_CD,amt_early,amt_late,change_rate(%),amt_diff
6,x,100337580000,181281480000,80.67,80943900000
5,6,531785970000,637183050000,19.82,105397080000
4,5,513835070000,556050980000,8.22,42215910000
2,3,271245650000,292180800000,7.72,20935150000
3,4,393825330000,420311970000,6.73,26486640000
1,2,157476330000,165229920000,4.92,7753590000
0,1,13728200000,12740890000,-7.19,-987310000


In [19]:
print(compare_korean_age.to_string(index=False))

AGE_CD    amt_early     amt_late  change_rate(%)     amt_diff
     1  13728200000  12740890000           -7.19   -987310000
     2 157476330000 165229920000            4.92   7753590000
     3 271245650000 292180800000            7.72  20935150000
     4 393825330000 420311970000            6.73  26486640000
     5 513835070000 556050980000            8.22  42215910000
     6 531785970000 637183050000           19.82 105397080000
     x 100337580000 181281480000           80.67  80943900000


In [20]:
# 60대 + 일반한식
sixties_korean = df[
    (df['AGE_CD'] == '6') &
    (df['TP_BUZ_NM'] == '일반한식')
]

# 1~2월
early_ccg = (
    sixties_korean[
        sixties_korean['STRD_YYMM'].isin([202601, 202602])
    ]
    .groupby('CCG_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)

# 5~6월
late_ccg = (
    sixties_korean[
        sixties_korean['STRD_YYMM'].isin([202605, 202606])
    ]
    .groupby('CCG_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)

# 비교
compare_korean_ccg = pd.merge(
    early_ccg,
    late_ccg,
    on='CCG_NM'
)

compare_korean_ccg['change_rate(%)'] = (
    (compare_korean_ccg['amt_late'] - compare_korean_ccg['amt_early'])
    / compare_korean_ccg['amt_early'] * 100
).round(2)

compare_korean_ccg['amt_diff'] = (
    compare_korean_ccg['amt_late']
    - compare_korean_ccg['amt_early']
)

# 증가액 TOP 15
display(
    compare_korean_ccg
    .sort_values('amt_diff', ascending=False)
    .head(15)
)

,CCG_NM,amt_early,amt_late,change_rate(%),amt_diff
181,중구,16609980000,19771570000,19.03,3161590000
61,동구,13220620000,15895640000,20.23,2675020000
85,북구,14880040000,17444780000,17.24,2564740000
92,서구,14237890000,16281100000,14.35,2043210000
47,남구,11136410000,12994060000,16.68,1857650000
178,제주시,8829250000,10309540000,16.77,1480290000
78,부산진구,7660720000,9042250000,18.03,1381530000
49,남양주시,5253550000,6593510000,25.51,1339960000
55,달서구,6367310000,7681160000,20.63,1313850000
28,광주시,3741730000,4976240000,32.99,1234510000


In [21]:
# 초기(1~2월) 소비금액 10억 원 이상인 시군구만 대상으로 증가율 확인
ccg_filtered = compare_korean_ccg[
    compare_korean_ccg['amt_early'] >= 1_000_000_000
]

display(
    ccg_filtered
    .sort_values('change_rate(%)', ascending=False)
    .head(15)
)

,CCG_NM,amt_early,amt_late,change_rate(%),amt_diff
131,여주시,1073440000,1846890000,72.05,773450000
129,양평군,1867510000,2898350000,55.20,1030840000
22,공주시,1614430000,2263850000,40.23,649420000
213,포천시,2291050000,3193930000,39.41,902880000
58,당진시,1489050000,2031380000,36.42,542330000
218,함안군,1030120000,1404980000,36.39,374860000
46,나주시,1091130000,1487820000,36.36,396690000
179,제천시,1380100000,1865170000,35.15,485070000
7,강화군,1344680000,1814350000,34.93,469670000
121,안성시,2090630000,2819880000,34.88,729250000


In [3]:
# B.수도권 / 지방 구분
capital_area = ['서울특별시', '경기도', '인천광역시']

df['region_type'] = df['SIDO_NM'].apply(
    lambda x: '수도권' if x in capital_area else '지방'
)

# 1~2월
early_region_age = (
    df[df['STRD_YYMM'].isin([202601, 202602])]
    .groupby(['region_type', 'AGE_CD'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)

# 5~6월
late_region_age = (
    df[df['STRD_YYMM'].isin([202605, 202606])]
    .groupby(['region_type', 'AGE_CD'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)

# 비교
compare_region_age = pd.merge(
    early_region_age,
    late_region_age,
    on=['region_type', 'AGE_CD']
)

compare_region_age['change_rate(%)'] = (
    (compare_region_age['amt_late'] - compare_region_age['amt_early'])
    / compare_region_age['amt_early'] * 100
).round(2)

compare_region_age['amt_diff'] = (
    compare_region_age['amt_late']
    - compare_region_age['amt_early']
)

display(compare_region_age)

,region_type,AGE_CD,amt_early,amt_late,change_rate(%),amt_diff
0,수도권,1,23840820000,25921290000,8.73,2080470000
1,수도권,2,255777450000,266906640000,4.35,11129190000
2,수도권,3,422196930000,439970770000,4.21,17773840000
3,수도권,4,534366960000,549910560000,2.91,15543600000
4,수도권,5,603646800000,625974210000,3.70,22327410000
5,수도권,6,625115000000,683995290000,9.42,58880290000
6,수도권,x,87554190000,111601260000,27.47,24047070000
7,지방,1,27109110000,29913640000,10.35,2804530000
8,지방,2,245456310000,263812490000,7.48,18356180000
9,지방,3,412811160000,436069200000,5.63,23258040000


In [4]:
# c. 편의점 + 대형할인점만 추출
channel_df = df[
    df['TP_BUZ_NM'].isin(['편 의 점', '대형할인점'])
]

# 1~2월
early_channel = (
    channel_df[
        channel_df['STRD_YYMM'].isin([202601, 202602])
    ]
    .groupby(['TP_BUZ_NM', 'AGE_CD'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)

# 5~6월
late_channel = (
    channel_df[
        channel_df['STRD_YYMM'].isin([202605, 202606])
    ]
    .groupby(['TP_BUZ_NM', 'AGE_CD'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)

# 비교
compare_channel_age = pd.merge(
    early_channel,
    late_channel,
    on=['TP_BUZ_NM', 'AGE_CD']
)

compare_channel_age['change_rate(%)'] = (
    (compare_channel_age['amt_late'] - compare_channel_age['amt_early'])
    / compare_channel_age['amt_early'] * 100
).round(2)

compare_channel_age['amt_diff'] = (
    compare_channel_age['amt_late']
    - compare_channel_age['amt_early']
)

display(
    compare_channel_age.sort_values(
        ['TP_BUZ_NM', 'change_rate(%)'],
        ascending=[True, False]
    )
)

,TP_BUZ_NM,AGE_CD,amt_early,amt_late,change_rate(%),amt_diff
6,대형할인점,x,16500510000,16871710000,2.25,371200000
5,대형할인점,6,174535780000,166367480000,-4.68,-8168300000
2,대형할인점,3,82266790000,72249200000,-12.18,-10017590000
4,대형할인점,5,148668100000,130515160000,-12.21,-18152940000
3,대형할인점,4,131353470000,113092620000,-13.90,-18260850000
1,대형할인점,2,28666710000,24313460000,-15.19,-4353250000
0,대형할인점,1,2064550000,1570810000,-23.92,-493740000
13,편 의 점,x,30129180000,65926810000,118.81,35797630000
7,편 의 점,1,14195910000,18757970000,32.14,4562060000
12,편 의 점,6,102877390000,126258000000,22.73,23380610000


In [5]:
# D. 연령대별 전체 소비금액
age_total = (
    df
    .groupby('AGE_CD')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'age_total'})
)

# 주요 업종만 추출
target_industries = ['편 의 점', '대형할인점', '슈퍼 마켓', '일반한식']

age_industry = (
    df[df['TP_BUZ_NM'].isin(target_industries)]
    .groupby(['AGE_CD', 'TP_BUZ_NM'])['amt']
    .sum()
    .reset_index()
)

# 연령대별 전체 소비금액과 결합
age_industry = age_industry.merge(
    age_total,
    on='AGE_CD'
)

# 해당 업종이 연령대 전체 소비에서 차지하는 비중
age_industry['share(%)'] = (
    age_industry['amt'] / age_industry['age_total'] * 100
).round(2)

# 연령대별 업종 구성비
age_share_pivot = age_industry.pivot(
    index='AGE_CD',
    columns='TP_BUZ_NM',
    values='share(%)'
)

display(age_share_pivot)

TP_BUZ_NM,대형할인점,슈퍼 마켓,일반한식,편 의 점
AGE_CD,,,,
1,3.24,5.94,24.31,31.54
2,5.08,8.17,31.21,22.09
3,8.99,11.15,32.91,18.59
4,10.44,14.01,34.71,16.60
5,10.08,17.04,38.94,13.40
6,11.93,22.24,41.23,8.12
x,4.76,15.00,37.93,12.67


In [6]:
# A결과 살릴지 버릴지
# 60대 데이터만 추출
sixties = df[df['AGE_CD'] == '6'].copy()

# 시군구별 60대 전체 소비금액
ccg_total = (
    sixties
    .groupby('CCG_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'total_amt'})
)

# 시군구별 60대 일반한식 소비금액
korean_food = (
    sixties[sixties['TP_BUZ_NM'] == '일반한식']
    .groupby('CCG_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'korean_food_amt'})
)

# 결합
ccg_korean_food = ccg_total.merge(
    korean_food,
    on='CCG_NM',
    how='left'
)

ccg_korean_food['korean_food_share(%)'] = (
    ccg_korean_food['korean_food_amt']
    / ccg_korean_food['total_amt']
    * 100
).round(2)

# 일반한식 소비 비중이 높은 지역
display(
    ccg_korean_food.sort_values(
        'korean_food_share(%)',
        ascending=False
    ).head(20)
)

,CCG_NM,total_amt,korean_food_amt,korean_food_share(%)
220,함평군,1747270000,1261430000,72.19
57,담양군,7887340000,5600160000,71.00
70,무주군,1372590000,917880000,66.87
185,진안군,1516670000,1008720000,66.51
21,곡성군,1646090000,1082740000,65.78
171,장성군,2753560000,1800880000,65.40
19,고창군,3731950000,2368280000,63.46
211,평창군,3970290000,2499360000,62.95
31,구례군,1989490000,1249160000,62.79
230,화순군,5658890000,3547460000,62.69


In [7]:
# 60대 일반한식 소비의 1~2월 vs 5~6월 비교
sixties_korean = df[
    (df['AGE_CD'] == '6') &
    (df['TP_BUZ_NM'] == '일반한식')
].copy()

early = (
    sixties_korean[
        sixties_korean['STRD_YYMM'].isin([202601, 202602])
    ]
    .groupby('CCG_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)

late = (
    sixties_korean[
        sixties_korean['STRD_YYMM'].isin([202605, 202606])
    ]
    .groupby('CCG_NM')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)

# 증가율 계산
korean_change = early.merge(
    late,
    on='CCG_NM'
)

korean_change['change_rate(%)'] = (
    (korean_change['amt_late'] - korean_change['amt_early'])
    / korean_change['amt_early'] * 100
).round(2)

korean_change['amt_diff'] = (
    korean_change['amt_late']
    - korean_change['amt_early']
)

# 앞에서 구한 일반한식 비중과 결합
final_korean = ccg_korean_food.merge(
    korean_change,
    on='CCG_NM',
    how='inner'
)

# 일반한식 비중 50% 이상 + 증가율 기준으로 정렬
result = final_korean[
    final_korean['korean_food_share(%)'] >= 50
].sort_values(
    'change_rate(%)',
    ascending=False
)

display(result)

,CCG_NM,total_amt,korean_food_amt,korean_food_share(%),amt_early,amt_late,change_rate(%),amt_diff
156,울릉군,1176320000,634210000,53.91,64950000,356630000,449.08,291680000
142,영월군,2332180000,1459140000,62.57,304050000,628450000,106.69,324400000
54,단양군,2867800000,1650060000,57.54,353360000,722740000,104.53,369380000
208,태안군,6486530000,3425370000,52.81,749820000,1445070000,92.72,695250000
21,곡성군,1646090000,1082740000,65.78,247220000,473440000,91.51,226220000
...,...,...,...,...,...,...,...,...
70,무주군,1372590000,917880000,66.87,308190000,355800000,15.45,47610000
180,종로구,31907810000,18769080000,58.82,5738450000,6566540000,14.43,828090000
158,울진군,3693760000,2155790000,58.36,721750000,737500000,2.18,15750000
197,청도군,5411610000,2971170000,54.90,947980000,928530000,-2.05,-19450000


In [8]:
# 분석할 업종만 추출
# C에서 확인한 '편 의 점'과 '대형할인점'만 분석한다.
channel_df = df[
    df['TP_BUZ_NM'].isin(['편 의 점', '대형할인점'])
].copy()


# 1~2월 소비금액 계산
# 비교의 기준이 되는 초기 소비 규모를 구한다.
early = (
    channel_df[
        channel_df['STRD_YYMM'].isin([202601, 202602])
    ]
    .groupby(['region_type', 'AGE_CD', 'TP_BUZ_NM'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)


# 5~6월 소비금액 계산
# 초기와 비교할 최근 소비 규모를 구한다.
late = (
    channel_df[
        channel_df['STRD_YYMM'].isin([202605, 202606])
    ]
    .groupby(['region_type', 'AGE_CD', 'TP_BUZ_NM'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)


# 초기와 최근 데이터를 하나의 표로 결합
compare_channel_region = pd.merge(
    early,
    late,
    on=['region_type', 'AGE_CD', 'TP_BUZ_NM']
)


# 1~2월 대비 5~6월의 소비 증감률 계산
compare_channel_region['change_rate(%)'] = (
    (compare_channel_region['amt_late'] - compare_channel_region['amt_early'])
    / compare_channel_region['amt_early']
    * 100
).round(2)


# 실제 소비금액이 얼마나 증가하거나 감소했는지 계산
compare_channel_region['amt_diff'] = (
    compare_channel_region['amt_late']
    - compare_channel_region['amt_early']
)


# 결과 확인
# 연령대와 업종별로 수도권/지방의 변화를 비교하기 쉽게 정렬한다.
display(
    compare_channel_region.sort_values(
        ['AGE_CD', 'TP_BUZ_NM', 'region_type']
    )
)

,region_type,AGE_CD,TP_BUZ_NM,amt_early,amt_late,change_rate(%),amt_diff
0,수도권,1,대형할인점,1081020000,813620000,-24.74,-267400000
14,지방,1,대형할인점,983530000,757190000,-23.01,-226340000
1,수도권,1,편 의 점,6676910000,8518390000,27.58,1841480000
15,지방,1,편 의 점,7519000000,10239580000,36.18,2720580000
2,수도권,2,대형할인점,16281410000,13692850000,-15.90,-2588560000
16,지방,2,대형할인점,12385300000,10620610000,-14.25,-1764690000
3,수도권,2,편 의 점,53105080000,60797350000,14.48,7692270000
17,지방,2,편 의 점,51449080000,61119530000,18.80,9670450000
4,수도권,3,대형할인점,47055920000,41294290000,-12.24,-5761630000
18,지방,3,대형할인점,35210870000,30954910000,-12.09,-4255960000


In [9]:
# D연령대별 전체 소비금액 계산
# 각 연령대의 소비 규모가 다르기 때문에,
# 업종별 금액이 아니라 '해당 연령대 안에서의 소비 비중'을 비교한다.
age_total = (
    df
    .groupby('AGE_CD')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'age_total'})
)


# 연령대 × 업종별 소비금액 계산
age_industry = (
    df
    .groupby(['AGE_CD', 'TP_BUZ_NM'])['amt']
    .sum()
    .reset_index()
)


# 연령대별 전체 소비금액을 결합
age_industry = age_industry.merge(
    age_total,
    on='AGE_CD'
)


# 각 연령대에서 해당 업종이 차지하는 소비 비중 계산
age_industry['share(%)'] = (
    age_industry['amt']
    / age_industry['age_total']
    * 100
).round(2)


# 연령대별로 소비 비중이 높은 업종 순서 확인
display(
    age_industry.sort_values(
        ['AGE_CD', 'share(%)'],
        ascending=[True, False]
    )
)

,AGE_CD,TP_BUZ_NM,amt,age_total,share(%)
8,1,편 의 점,51900210000,164574750000,31.54
4,1,일반한식,40006090000,164574750000,24.31
1,1,서양음식,32929440000,164574750000,20.01
2,1,슈퍼 마켓,9778660000,164574750000,5.94
3,1,스넥,7961190000,164574750000,4.84
...,...,...,...,...,...
68,x,스넥,40840270000,1056216910000,3.87
72,x,중국음식,33484790000,1056216910000,3.17
70,x,일식회집,30253250000,1056216910000,2.86
64,x,갈비전문점,562600000,1056216910000,0.05


In [10]:
# 비교할 기간을 1~2월과 5~6월로 나눈다.
# 앞에서 사용한 기간과 동일하게 맞춰서 결과를 비교한다.
early_df = df[
    df['STRD_YYMM'].isin([202601, 202602])
].copy()

late_df = df[
    df['STRD_YYMM'].isin([202605, 202606])
].copy()


# 1~2월 연령대 × 업종별 소비금액
early_industry = (
    early_df
    .groupby(['AGE_CD', 'TP_BUZ_NM'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)


# 5~6월 연령대 × 업종별 소비금액
late_industry = (
    late_df
    .groupby(['AGE_CD', 'TP_BUZ_NM'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)


# 1~2월과 5~6월의 소비금액을 하나의 표로 결합한다.
compare_industry = pd.merge(
    early_industry,
    late_industry,
    on=['AGE_CD', 'TP_BUZ_NM'],
    how='outer'
).fillna(0)


# 각 연령대의 전체 소비금액을 계산한다.
early_total = (
    early_df
    .groupby('AGE_CD')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'total_early'})
)

late_total = (
    late_df
    .groupby('AGE_CD')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'total_late'})
)


# 각 연령대의 전체 소비금액을 결합한다.
compare_industry = compare_industry.merge(
    early_total,
    on='AGE_CD'
)

compare_industry = compare_industry.merge(
    late_total,
    on='AGE_CD'
)


# 1~2월에서 해당 업종이 차지했던 비중을 계산한다.
compare_industry['share_early(%)'] = (
    compare_industry['amt_early']
    / compare_industry['total_early']
    * 100
).round(2)


# 5~6월에서 해당 업종이 차지하는 비중을 계산한다.
compare_industry['share_late(%)'] = (
    compare_industry['amt_late']
    / compare_industry['total_late']
    * 100
).round(2)


# 소비구조가 몇 %p 변했는지 계산한다.
# 예: 10% → 12%라면 +2%p
compare_industry['share_change(%p)'] = (
    compare_industry['share_late(%)']
    - compare_industry['share_early(%)']
).round(2)


# 연령대별로 업종 비중 변화가 큰 순서로 정렬한다.
display(
    compare_industry.sort_values(
        ['AGE_CD', 'share_change(%p)'],
        ascending=[True, False]
    )
)

,AGE_CD,TP_BUZ_NM,amt_early,amt_late,total_early,total_late,share_early(%),share_late(%),share_change(%p)
8,1,편 의 점,14195910000,18757970000,50949930000,55834930000,27.86,33.60,5.74
1,1,서양음식,9988600000,11286530000,50949930000,55834930000,19.60,20.21,0.61
3,1,스넥,2357720000,2706030000,50949930000,55834930000,4.63,4.85,0.22
7,1,중국음식,2160350000,2344780000,50949930000,55834930000,4.24,4.20,-0.04
6,1,제 과 점,1369920000,1361770000,50949930000,55834930000,2.69,2.44,-0.25
...,...,...,...,...,...,...,...,...,...
72,x,중국음식,9171920000,14328240000,258453690000,489143610000,3.55,2.93,-0.62
70,x,일식회집,8376560000,12253590000,258453690000,489143610000,3.24,2.51,-0.73
69,x,일반한식,100337580000,181281480000,258453690000,489143610000,38.82,37.06,-1.76
66,x,서양음식,39762820000,64246720000,258453690000,489143610000,15.38,13.13,-2.25


In [12]:
# X를 제외하고 실제 연령대 1~6만 확인
# 각 연령대에서 소비 비중 변화가 큰 업종부터 정렬
display(
    compare_industry[
        compare_industry['AGE_CD'].isin(['1', '2', '3', '4', '5', '6'])
    ]
    .sort_values(
        ['AGE_CD', 'share_change(%p)'],
        ascending=[True, False]
    )
)
# 모든 행과 모든 열을 생략하지 않고 표시
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

display(
    compare_industry.sort_values(
        ['AGE_CD', 'share_change(%p)'],
        ascending=[True, False]
    )
)

,AGE_CD,TP_BUZ_NM,amt_early,amt_late,total_early,total_late,share_early(%),share_late(%),share_change(%p)
8,1,편 의 점,14195910000,18757970000,50949930000,55834930000,27.86,33.60,5.74
1,1,서양음식,9988600000,11286530000,50949930000,55834930000,19.60,20.21,0.61
3,1,스넥,2357720000,2706030000,50949930000,55834930000,4.63,4.85,0.22
7,1,중국음식,2160350000,2344780000,50949930000,55834930000,4.24,4.20,-0.04
6,1,제 과 점,1369920000,1361770000,50949930000,55834930000,2.69,2.44,-0.25
...,...,...,...,...,...,...,...,...,...
63,6,한정식,151000000,214120000,1344564010000,1506136280000,0.01,0.01,0.00
61,6,중국음식,33740510000,36031320000,1344564010000,1506136280000,2.51,2.39,-0.12
60,6,제 과 점,37484290000,38629740000,1344564010000,1506136280000,2.79,2.56,-0.23
54,6,대형할인점,174535780000,166367480000,1344564010000,1506136280000,12.98,11.05,-1.93


,AGE_CD,TP_BUZ_NM,amt_early,amt_late,total_early,total_late,share_early(%),share_late(%),share_change(%p)
8,1,편 의 점,14195910000,18757970000,50949930000,55834930000,27.86,33.60,5.74
1,1,서양음식,9988600000,11286530000,50949930000,55834930000,19.60,20.21,0.61
3,1,스넥,2357720000,2706030000,50949930000,55834930000,4.63,4.85,0.22
7,1,중국음식,2160350000,2344780000,50949930000,55834930000,4.24,4.20,-0.04
6,1,제 과 점,1369920000,1361770000,50949930000,55834930000,2.69,2.44,-0.25
2,1,슈퍼 마켓,3186610000,3304620000,50949930000,55834930000,6.25,5.92,-0.33
5,1,일식회집,1898070000,1761530000,50949930000,55834930000,3.73,3.15,-0.58
0,1,대형할인점,2064550000,1570810000,50949930000,55834930000,4.05,2.81,-1.24
4,1,일반한식,13728200000,12740890000,50949930000,55834930000,26.94,22.82,-4.12
18,2,편 의 점,104554160000,121916880000,501233760000,530719130000,20.86,22.97,2.11


In [13]:
# ≤20대 + 편의점 데이터만 추출
young_convenience = df[
    (df['AGE_CD'] == '1') &
    (df['TP_BUZ_NM'] == '편 의 점')
].copy()


# 시군구별 1~2월 소비금액 계산
early = (
    young_convenience[
        young_convenience['STRD_YYMM'].isin([202601, 202602])
    ]
    .groupby(['SIDO_NM', 'CCG_NM'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_early'})
)


# 시군구별 5~6월 소비금액 계산
late = (
    young_convenience[
        young_convenience['STRD_YYMM'].isin([202605, 202606])
    ]
    .groupby(['SIDO_NM', 'CCG_NM'])['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'amt_late'})
)


# 1~2월과 5~6월 데이터를 하나로 합친다.
young_convenience_region = pd.merge(
    early,
    late,
    on=['SIDO_NM', 'CCG_NM'],
    how='inner'
)


# 시군구별 소비금액 변화율 계산
young_convenience_region['change_rate(%)'] = (
    (young_convenience_region['amt_late']
     - young_convenience_region['amt_early'])
    / young_convenience_region['amt_early']
    * 100
).round(2)


# 너무 작은 금액에서 나온 극단적인 변화율을 피하기 위해
# 1~2월 소비금액이 1억원 이상인 지역만 확인한다.
result = young_convenience_region[
    young_convenience_region['amt_early'] >= 100_000_000
].sort_values(
    'change_rate(%)',
    ascending=False
)


display(result)

,SIDO_NM,CCG_NM,amt_early,amt_late,change_rate(%)
87,경상북도,경산시,143860000,305530000,112.38
236,충청남도,천안시 동남구,110140000,194600000,76.68
156,서울특별시,동작구,104060000,173970000,67.18
234,충청남도,아산시,114140000,189790000,66.28
183,인천광역시,연수구,107780000,177530000,64.72
113,광주광역시,북구,109170000,174730000,60.05
150,서울특별시,광진구,102420000,161670000,57.85
155,서울특별시,동대문구,100530000,156890000,56.06
125,대전광역시,동구,103720000,159980000,54.24
120,대구광역시,북구,178860000,275670000,54.13


In [14]:
# 대형할인점 데이터만 추출
large_mart = df[df['TP_BUZ_NM'] == '대형할인점'].copy()

# 5~6월 기준 연령별 대형할인점 소비금액
mart_late = (
    large_mart[large_mart['STRD_YYMM'].isin([202605, 202606])]
    .groupby('AGE_CD')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'large_mart_amt_late'})
)

# 5~6월 기준 연령별 전체 소비금액
total_late = (
    df[df['STRD_YYMM'].isin([202605, 202606])]
    .groupby('AGE_CD')['amt']
    .sum()
    .reset_index()
    .rename(columns={'amt': 'total_amt_late'})
)

# 두 데이터를 합치기
age_share = pd.merge(
    mart_late,
    total_late,
    on='AGE_CD',
    how='inner'
)

# 전체 소비 중 대형할인점 비중 계산
age_share['large_mart_share(%)'] = (
    age_share['large_mart_amt_late']
    / age_share['total_amt_late']
    * 100
).round(2)

# 보기 좋게 정렬
age_share.sort_values('AGE_CD')

,AGE_CD,large_mart_amt_late,total_amt_late,large_mart_share(%)
0,1,1570810000,55834930000,2.81
1,2,24313460000,530719130000,4.58
2,3,72249200000,876039970000,8.25
3,4,113092620000,1187651410000,9.52
4,5,130515160000,1406497990000,9.28
5,6,166367480000,1506136280000,11.05
6,x,16871710000,489143610000,3.45


In [16]:
target = df[(df['AGE_CD'].isin(['4','5'])) & (df['TP_BUZ_NM']=='대형할인점')]
monthly = target.groupby(['STRD_YYMM','AGE_CD'])['amt'].sum().unstack()
print(monthly)

AGE_CD               4            5
STRD_YYMM                          
202601     67585280000  76466040000
202602     63768190000  72202060000
202603     62258160000  69303940000
202604     57435320000  66065950000
202605     62296140000  70325710000
202606     50796480000  60189450000


In [17]:
# 실제 연령대만 사용 (X 제외)
df = df[df['AGE_CD'].isin(['1','2','3','4','5','6'])]

months = [202601, 202602, 202603, 202604, 202605, 202606]
age_map = {'1':'20대이하','2':'20대','3':'30대','4':'40대','5':'50대','6':'60대'}

results = []

for (age, cat), g in df.groupby(['AGE_CD', 'TP_BUZ_NM']):
    monthly = g.groupby('STRD_YYMM')['amt'].sum().reindex(months)

    # 6개월 데이터가 다 있는 조합만 (표본 안정성 최소 조건)
    if monthly.isna().any():
        continue

    # ① 1월→6월 변화율
    change_rate = (monthly[202606] - monthly[202601]) / monthly[202601] * 100

    # ② 절대 감소액 (1월 대비 6월)
    abs_change = monthly[202606] - monthly[202601]

    # ⑤ 표본 규모 프록시: 6개월 평균 소비금액
    avg_amt = monthly.mean()

    # ⑥ 지속성 점수: 전월 대비 하락한 전환 횟수 / 5
    diffs = monthly.diff().dropna()
    persistence = (diffs < 0).sum() / len(diffs)

    results.append({
        '연령': age_map[age],
        '업종': cat,
        '변화율(%)': round(change_rate, 2),
        '절대변화액': round(abs_change, 0),
        '평균월소비금액': round(avg_amt, 0),
        '지속성점수': round(persistence, 2),
    })

score_df = pd.DataFrame(results)

# ④ 연령 내 업종 비중 (해당 연령의 전체 소비 대비)
age_total = score_df.groupby('연령')['평균월소비금액'].transform('sum')
score_df['업종비중(%)'] = round(score_df['평균월소비금액'] / age_total * 100, 2)

# 위축 세그먼트만 필터 (감소 + 지속성 있는 것만)
declining = score_df[
    (score_df['변화율(%)'] < 0) &
    (score_df['지속성점수'] >= 0.6)   # 5번 전환 중 최소 3번 이상 하락
].copy()

# 종합 스코어: 절대변화액(가장 중요) + 지속성 가중
declining['종합점수'] = (
    declining['절대변화액'].rank(ascending=True) * 0.5 +   # 많이 줄어든 순(음수라 오름차순=감소폭 큰 순)
    declining['지속성점수'].rank(ascending=False) * 0.3 +
    declining['변화율(%)'].rank(ascending=True) * 0.2
)

declining = declining.sort_values('종합점수').reset_index(drop=True)

print(declining.to_string(index=False))

   연령    업종  변화율(%)        절대변화액      평균월소비금액  지속성점수  업종비중(%)  종합점수
  40대 대형할인점  -24.84 -16788800000 6.068993e+10    0.8    10.44  1.95
  50대 대형할인점  -21.29 -16276590000 6.909219e+10    0.8    10.08  3.25
  30대 대형할인점  -23.35  -9904060000 3.839812e+10    0.8     8.99  3.35
  20대 대형할인점  -24.58  -3598480000 1.315656e+10    0.8     5.08  6.15
  40대 슈퍼 마켓  -11.27  -9854220000 8.140189e+10    0.8    14.01  6.55
  50대 슈퍼 마켓   -7.70  -9378200000 1.168656e+11    0.8    17.04  9.05
  60대 대형할인점  -11.02  -9786030000 8.518848e+10    0.6    11.93 12.20
  40대  중국음식  -13.14  -2727570000 1.967050e+10    0.6     3.39 13.30
  40대 제 과 점  -14.92  -2578970000 1.649430e+10    0.6     2.84 13.40
  20대  일반한식   -7.94  -6528020000 8.087259e+10    0.6    31.21 13.80
  30대 슈퍼 마켓   -9.03  -4543530000 4.764504e+10    0.6    11.15 13.90
20대이하  일반한식  -23.25  -1699920000 6.667682e+09    0.6    24.31 14.70
  30대 제 과 점  -14.31  -1814630000 1.211883e+10    0.6     2.84 15.10
  50대 제 과 점  -11.69  -2156220000 1.798340e+10   

In [18]:
months = [202601, 202602, 202603, 202604, 202605, 202606]
age_map = {'1':'20대이하','2':'20대','3':'30대','4':'40대','5':'50대','6':'60대'}

results = []
for (age, cat), g in df.groupby(['AGE_CD', 'TP_BUZ_NM']):
    monthly = g.groupby('STRD_YYMM')['amt'].sum().reindex(months)
    if monthly.isna().any():
        continue

    change_rate = (monthly[202606] - monthly[202601]) / monthly[202601] * 100
    abs_change = monthly[202606] - monthly[202601]
    avg_amt = monthly.mean()
    diffs = monthly.diff().dropna()
    persistence = (diffs < 0).sum() / len(diffs)

    results.append({
        '연령': age_map[age], '업종': cat,
        '변화율(%)': round(change_rate, 2),
        '절대변화액': round(abs_change, 0),
        '평균월소비금액': round(avg_amt, 0),
        '지속성점수': round(persistence, 2),
    })

score_df = pd.DataFrame(results)
age_total = score_df.groupby('연령')['평균월소비금액'].transform('sum')
score_df['업종비중(%)'] = round(score_df['평균월소비금액'] / age_total * 100, 2)

# ⑤ 최소 규모 필터: 평균월소비금액 10억 미만 제외 (노이즈 컷)
MIN_SCALE = 1e9

declining = score_df[
    (score_df['변화율(%)'] < 0) &
    (score_df['지속성점수'] >= 0.6) &
    (score_df['평균월소비금액'] >= MIN_SCALE)
].copy()

declining['종합점수'] = (
    declining['절대변화액'].rank(ascending=True) * 0.5 +
    declining['지속성점수'].rank(ascending=False) * 0.3 +
    declining['변화율(%)'].rank(ascending=True) * 0.2
)

declining = declining.sort_values('종합점수').reset_index(drop=True)
print(declining.to_string(index=False))

   연령    업종  변화율(%)        절대변화액      평균월소비금액  지속성점수  업종비중(%)  종합점수
  40대 대형할인점  -24.84 -16788800000 6.068993e+10    0.8    10.44  1.75
  50대 대형할인점  -21.29 -16276590000 6.909219e+10    0.8    10.08  3.05
  30대 대형할인점  -23.35  -9904060000 3.839812e+10    0.8     8.99  3.15
  40대 슈퍼 마켓  -11.27  -9854220000 8.140189e+10    0.8    14.01  5.75
  20대 대형할인점  -24.58  -3598480000 1.315656e+10    0.8     5.08  5.95
  50대 슈퍼 마켓   -7.70  -9378200000 1.168656e+11    0.8    17.04  8.05
  60대 대형할인점  -11.02  -9786030000 8.518848e+10    0.6    11.93 10.45
  40대  중국음식  -13.14  -2727570000 1.967050e+10    0.6     3.39 11.75
  40대 제 과 점  -14.92  -2578970000 1.649430e+10    0.6     2.84 11.85
  20대  일반한식   -7.94  -6528020000 8.087259e+10    0.6    31.21 12.05
  30대 슈퍼 마켓   -9.03  -4543530000 4.764504e+10    0.6    11.15 12.15
  30대 제 과 점  -14.31  -1814630000 1.211883e+10    0.6     2.84 13.55
  50대 제 과 점  -11.69  -2156220000 1.798340e+10    0.6     2.62 13.65
  50대  중국음식  -11.27  -2353230000 1.993208e+10   